In [10]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# =========================================================
# 1. الاستخراج (Extract) والربط بالسيرفر المصدر
# =========================================================
print("1. جاري سحب البيانات الخام من Sales_DB...")

# استخدام اسم السيرفر الخاص بجهازك وحرف r للتعامل مع الشرطة المائلة \
source_conn_str = (
    r"mssql+pyodbc://DESKTOP-95SCKC5\SQLEXPRESS/Sales_DB?"
    "driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)
source_engine = create_engine(source_conn_str)

# سحب الجدول الخام
df = pd.read_sql("SELECT * FROM Raw_Orders", source_engine)

print("--- البيانات قبل التنظيف ---")
print(df)


# =========================================================
# 2. المعالجة والتنظيف (Transform)
# =========================================================
print("\n2. جاري تنظيف وتجهيز البيانات...")

# أ) إزالة التكرارات
df = df.drop_duplicates(subset=['Order_Code'])

# ب) تنظيف اسم العميل (حذف المسافات الحروف الغريبة وتنسيق الاسم)
df['Customer_Info'] = df['Customer_Info'].fillna('Unknown')
df['Customer_Name'] = df['Customer_Info'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True).str.strip().str.title()

# ج) تنظيف الأسعار (حذف $ و EGP وحساب القيمة المطلقة)
df['Price_Tag'] = df['Price_Tag'].str.replace(r'[^\d.]', '', regex=True)
df['Clean_Price'] = pd.to_numeric(df['Price_Tag'], errors='coerce').fillna(0).abs()

# د) تنظيف الكميات وتحويلها لأرقام صحيحة
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce').fillna(1).astype(int)

# هـ) توحيد صيغة التواريخ والمدن
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce').dt.date
df['City'] = df['City'].str.strip().str.capitalize().fillna('Unknown')
df['City'] = df['City'].replace({'Alex': 'Alexandria'})

# و) حساب الإجمالي (Total Amount)
df['Total_Amount'] = df['Clean_Price'] * df['Quantity']

# إعادة ترتيب وتسمية الأعمدة لتطابق جدول الـ Data Warehouse
final_df = df[['Order_Code', 'Customer_Name', 'Order_Date', 'Clean_Price', 'Quantity', 'Total_Amount', 'City']].copy()
final_df.columns = ['Order_ID', 'Customer_Name', 'Order_Date', 'Clean_Price', 'Quantity', 'Total_Amount', 'Clean_City']

print("\n--- البيانات بعد التنظيف ---")
print(final_df)


# =========================================================
# 3. التحميل (Load) للـ Data Warehouse
# =========================================================
print("\n3. جاري إرسال البيانات إلى Sales_DW...")

target_conn_str = (
    r"mssql+pyodbc://DESKTOP-95SCKC5\SQLEXPRESS/Sales_DW?"
    "driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)
target_engine = create_engine(target_conn_str)

# رفع الجدول المنظف لـ Fact_Sales
final_df.to_sql('Fact_Sales', target_engine, if_exists='append', index=False)

print("\n🚀 تم بنجاح! البيانات استقرت في الـ Data Warehouse.")

1. جاري سحب البيانات الخام من Sales_DB...
--- البيانات قبل التنظيف ---
   Raw_ID Order_Code    Customer_Info    Order_Date  Price_Tag Quantity  \
0       1    ORD-101      ahmed ali      2026-01-15  $1,500.00        2   
1       2    ORD-102    MONA mahmoud!    15/01/2026    800 EGP        1   
2       3    ORD-101      ahmed ali      2026-01-15  $1,500.00        2   
3       4    ORD-103       Tarek Omar  Jan 16, 2026    -450.00        3   
4       5    ORD-104     UNKNOWN_USER    2026/01/17       FREE     NULL   
5       6    ORD-105    sara mohamed     2026-01-18  $3,200.50        5   
6       7    ORD-106              NaN    2026-01-19    $900.00        1   

           City  
0         cairo  
1        Cairo   
2         cairo  
3          ALEX  
4          Giza  
5   alexandria   
6           NaN  

2. جاري تنظيف وتجهيز البيانات...

--- البيانات بعد التنظيف ---
  Order_ID Customer_Name  Order_Date  Clean_Price  Quantity  Total_Amount  \
0  ORD-101     Ahmed Ali  2026-01-15       